# HƯỚNG DẪN HUẤN LUYỆN YOLOv8 NHẬN DIỆN BIỂN SỐ CHO VINHGATE (ĐẠI HỌC VINH)
Thực hiện trên Google Colab với GPU Tesla T4 miễn phí.

### CELL 1: Kiểm tra GPU Tesla T4, kết nối Google Drive và cài đặt thư viện

In [ ]:
# 1. Kiểm tra xem Google Colab đã cấp GPU Tesla T4 chưa
!nvidia-smi

# 2. Cài đặt thư viện Ultralytics (YOLOv8) và ONNX để xuất model
!pip install -q ultralytics onnx onnxruntime-gpu roboflow

# 3. Kết nối (Mount) với Google Drive để lưu kết quả an toàn không bị mất khi tắt máy
from google.colab import drive
drive.mount('/content/drive')

### CELL 2: Giải nén bộ dữ liệu và tạo file cấu hình data.yaml

In [ ]:
import os, yaml
# Giải nén file dataset.zip đã upload lên Google Drive vào ổ cứng ảo của Colab
!mkdir -p /content/datasets/vinhgate_lp
!unzip -q "/content/drive/MyDrive/VinhGate_Dataset/vinhgate_dataset.zip" -d /content/datasets/vinhgate_lp

# Tạo file cấu hình data.yaml chuẩn xác cho YOLOv8
data_config = {
    'path': '/content/datasets/vinhgate_lp',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 3,
    'names': ['license_plate', 'motorcycle', 'car']
}
yaml_path = '/content/datasets/vinhgate_lp/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)
print("Đã tạo thành công file cấu hình:", yaml_path)

### CELL 3: Huấn luyện mô hình (Fine-tune) từ Pretrained Weights yolov8n.pt

In [ ]:
from ultralytics import YOLO
# Tải mô hình pretrained yolov8n.pt
model = YOLO('yolov8n.pt')

# Bắt đầu quá trình Fine-tune và lưu trực tiếp kết quả vào Google Drive
results = model.train(
    data='/content/datasets/vinhgate_lp/data.yaml',
    epochs=50,                  # 50 vòng lặp là đủ hội tụ khi fine-tune từ yolov8n.pt
    imgsz=640,                  # Kích thước ảnh đầu vào chuẩn 640x640
    batch=16,                   # Batch size tối ưu cho GPU T4 16GB
    patience=10,                # Dừng sớm nếu 10 epoch liên tiếp không tăng độ chính xác
    optimizer='AdamW',          # Thuật toán tối ưu hội tụ nhanh
    lr0=0.001,
    project='/content/drive/MyDrive/VinhGate_Models',
    name='yolov8n_vinhgate_lp',
    exist_ok=True
)

### CELL 4: Đánh giá chỉ số (mAP50, Precision, Recall) để đưa vào Báo cáo

In [ ]:
from ultralytics import YOLO
best_model_path = '/content/drive/MyDrive/VinhGate_Models/yolov8n_vinhgate_lp/weights/best.pt'
best_model = YOLO(best_model_path)

# Chạy đánh giá độc lập trên tập Test
metrics = best_model.val(data='/content/datasets/vinhgate_lp/data.yaml', split='test')
print("================ KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH ================")
print(f"Precision (Độ chuẩn xác):     {metrics.box.mp:.4f}")
print(f"Recall (Độ phủ):              {metrics.box.mr:.4f}")
print(f"mAP@0.5 (Chỉ số chính):       {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 (Chỉ số nghiêm): {metrics.box.map:.4f}")

### CELL 5: Chạy thử nghiệm (Inference) trên cả ẢNH TĨNH và VIDEO

In [ ]:
# 1. Test trên thư mục ảnh
best_model.predict(
    source='/content/datasets/vinhgate_lp/test/images',
    conf=0.45,
    save=True,
    project='/content/drive/MyDrive/VinhGate_Models/predictions',
    name='test_images_result'
)

# 2. Test trên file VIDEO quay tại cổng ĐH Vinh
best_model.predict(
    source='/content/drive/MyDrive/VinhGate_Dataset/demo_cong_dh_vinh.mp4',
    conf=0.45,
    save=True,
    project='/content/drive/MyDrive/VinhGate_Models/predictions',
    name='test_video_result'
)
print("Đã xuất ảnh và video kết quả vào Google Drive!")

### CELL 6: Xuất mô hình sang chuẩn ONNX để chạy trên CPU laptop

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=640, simplify=True)
print("File ONNX đã sẵn sàng tại:", onnx_path)
# Tải 2 file best.pt và best.onnx về đặt vào thư mục VinhGate/models/